# Activation Patching

Reproduce the tiny clean/corrupt patching run from the repo root:

```bash
uv run python scripts/run_activation_patching.py --config configs/gpt2_small_clean_corrupt_tiny.yaml
```

Reproduce the controlled patching follow-up after a controls run exists:

```bash
uv run python scripts/select_controlled_patching_candidates.py --run runs/<controlled_run_id>
uv run python scripts/run_controlled_patching.py --config configs/gpt2_small_induction_controls.yaml --run runs/<controlled_run_id> --candidates runs/<controlled_run_id>/controlled_patching_candidates.csv --examples-per-family 8
uv run python scripts/summarize_run.py --run runs/<controlled_run_id>
```

This notebook only loads script artifacts for inspection. Layer-level `attn_out` patching is not head-specific unless an artifact records `head_specific_patch=true`.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

controlled_runs = sorted(
    p for p in Path("runs").glob("*_gpt2_small_induction_controls") if p.is_dir()
)
clean_corrupt_runs = sorted(
    p for p in Path("runs").glob("*_gpt2_small_clean_corrupt_tiny") if p.is_dir()
)
controlled_run = controlled_runs[-1] if controlled_runs else None
clean_corrupt_run = clean_corrupt_runs[-1] if clean_corrupt_runs else None
controlled_run, clean_corrupt_run

## Tiny Clean/Corrupt Patching

In [ ]:
if clean_corrupt_run and (clean_corrupt_run / "patching_results.csv").exists():
    display(pd.read_csv(clean_corrupt_run / "patching_results.csv"))
    figure = clean_corrupt_run / "figures" / "patching_heatmap.png"
    if figure.exists():
        display(Image(filename=str(figure)))
else:
    print("Missing tiny clean/corrupt patching artifacts.")

## Controlled Patching Follow-up

In [ ]:
if controlled_run:
    for filename in [
        "controlled_patching_results.csv",
        "controlled_patching_by_family.csv",
        "controlled_patching_by_candidate.csv",
    ]:
        path = controlled_run / filename
        if path.exists():
            print(filename)
            display(pd.read_csv(path))
        else:
            print(f"Missing {filename}")
else:
    print("No controlled induction run found.")

In [ ]:
if controlled_run:
    summary = controlled_run / "controlled_patching_summary.json"
    if summary.exists():
        print(summary.read_text())
    else:
        print("Missing controlled_patching_summary.json")

    for figure_name in [
        "controlled_patching_by_family.png",
        "controlled_patching_candidate_gap.png",
    ]:
        figure = controlled_run / "figures" / figure_name
        if figure.exists():
            display(Image(filename=str(figure)))
        else:
            print(f"Missing {figure_name}")